In [1]:
%pip install pandas scikit-learn joblib imbalanced-learn matplotlib seaborn category_encoders nevergrad numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# drop unknown EDA

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE

df = pd.read_csv("./bank-additional/bank-additional-full.csv",delimiter=";")
df.head()


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


#### drop yg aneh2

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

In [4]:
# df.drop(['nr.employed','euribor3m','cons.conf.idx','cons.price.idx','emp.var.rate'],axis=1,inplace=True)
df.drop(['emp.var.rate','contact','housing','loan','previous','marital'],axis=1,inplace=True)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   education       41188 non-null  object 
 3   default         41188 non-null  object 
 4   month           41188 non-null  object 
 5   day_of_week     41188 non-null  object 
 6   duration        41188 non-null  int64  
 7   campaign        41188 non-null  int64  
 8   pdays           41188 non-null  int64  
 9   poutcome        41188 non-null  object 
 10  cons.price.idx  41188 non-null  float64
 11  cons.conf.idx   41188 non-null  float64
 12  euribor3m       41188 non-null  float64
 13  nr.employed     41188 non-null  float64
 14  y               41188 non-null  object 
dtypes: float64(4), int64(4), object(7)
memory usage: 4.7+ MB


#### genapin duration

In [6]:
df['duration'] = (df['duration']/60).round()

#### drop column

In [7]:
df.drop(['day_of_week','default','pdays'],axis=1,inplace=True)
df.head()

,age,job,education,month,duration,campaign,poutcome,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,basic.4y,may,4.0,1,nonexistent,93.994,-36.4,4.857,5191.0,no
1,57,services,high.school,may,2.0,1,nonexistent,93.994,-36.4,4.857,5191.0,no
2,37,services,high.school,may,4.0,1,nonexistent,93.994,-36.4,4.857,5191.0,no
3,40,admin.,basic.6y,may,3.0,1,nonexistent,93.994,-36.4,4.857,5191.0,no
4,56,services,high.school,may,5.0,1,nonexistent,93.994,-36.4,4.857,5191.0,no


#### drop unknown

In [8]:
df = df[df['duration'] <= 35]

In [9]:
df = df[df['education'] != 'unknown']


In [10]:
df = df[df['campaign'] <= 30]

In [11]:
# df = df[df['housing'] != 'unknown']

In [12]:
df = df[df['job'] != 'unknown']

In [13]:
# df = df[df['marital'] != 'unknown']

#### feature age = bining & ordinal

In [14]:
'''
min_age = df['age'].min()   # 17
max_age = df['age'].max()   # 98

# 5‑year buckets: [17–21], [22–26], … , [87–91] (last bucket includes all >90)
bin_edges = list(range(min_age, max_age + 1, 5))      # e.g. [17, 22, 27, ..., 93]
# add an upper bound that is bigger than any age so the last value falls in
bin_edges.append(float('inf'))

# Labels for readability (optional)
labels = [f'{l}-{r-1}' for l, r in zip(bin_edges[:-2], bin_edges[1:-1])]
labels.append(f'{bin_edges[-2]}+')

# ── Apply cut ───────────────────────────────────────────
df['age_bin'] = pd.cut(df['age'],
                       bins=bin_edges,
                       labels=labels,
                       right=False,          # left‑inclusive [l,r)
                       include_lowest=True)  # include the very first value

# ── Ordinal encoding (1 = first bin, 2 = second …) ───────
df['age_ordinal'] = df['age_bin'].cat.codes + 1   # .codes gives 0‑based int

df.head()
'''


"\nmin_age = df['age'].min()   # 17\nmax_age = df['age'].max()   # 98\n\n# 5‑year buckets: [17–21], [22–26], … , [87–91] (last bucket includes all >90)\nbin_edges = list(range(min_age, max_age + 1, 5))      # e.g. [17, 22, 27, ..., 93]\n# add an upper bound that is bigger than any age so the last value falls in\nbin_edges.append(float('inf'))\n\n# Labels for readability (optional)\nlabels = [f'{l}-{r-1}' for l, r in zip(bin_edges[:-2], bin_edges[1:-1])]\nlabels.append(f'{bin_edges[-2]}+')\n\n# ── Apply cut ───────────────────────────────────────────\ndf['age_bin'] = pd.cut(df['age'],\n                       bins=bin_edges,\n                       labels=labels,\n                       right=False,          # left‑inclusive [l,r)\n                       include_lowest=True)  # include the very first value\n\n# ── Ordinal encoding (1\u202f=\u202ffirst bin, 2\u202f=\u202fsecond …) ───────\ndf['age_ordinal'] = df['age_bin'].cat.codes + 1   # .codes gives 0‑based int\n\ndf.head()\n"

In [15]:
# drop obsolete age feature :
# df.drop(['age','age_bin'],axis = 1,inplace=True)

In [16]:
# Convert y from yes/no to 0/1
df['y'] = df['y'].map({'yes': 1, 'no': 0})

#### contact binary encoder

In [17]:
from category_encoders import BinaryEncoder,OneHotEncoder,TargetEncoder


#encoder = BinaryEncoder(cols=['contact'])
#df = encoder.fit_transform(df)

#### education OHE

In [18]:
ohe = TargetEncoder(cols=['education'])
df = ohe.fit_transform(df,df['y'])

df.head()

,age,job,education,month,duration,campaign,poutcome,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,0.102477,may,4.0,1,nonexistent,93.994,-36.4,4.857,5191.0,0
1,57,services,0.108162,may,2.0,1,nonexistent,93.994,-36.4,4.857,5191.0,0
2,37,services,0.108162,may,4.0,1,nonexistent,93.994,-36.4,4.857,5191.0,0
3,40,admin.,0.082047,may,3.0,1,nonexistent,93.994,-36.4,4.857,5191.0,0
4,56,services,0.108162,may,5.0,1,nonexistent,93.994,-36.4,4.857,5191.0,0


#### housing binary

In [19]:
#encoder2 = BinaryEncoder(cols=['housing'])
#df = encoder2.fit_transform(df)

df.head()

,age,job,education,month,duration,campaign,poutcome,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,0.102477,may,4.0,1,nonexistent,93.994,-36.4,4.857,5191.0,0
1,57,services,0.108162,may,2.0,1,nonexistent,93.994,-36.4,4.857,5191.0,0
2,37,services,0.108162,may,4.0,1,nonexistent,93.994,-36.4,4.857,5191.0,0
3,40,admin.,0.082047,may,3.0,1,nonexistent,93.994,-36.4,4.857,5191.0,0
4,56,services,0.108162,may,5.0,1,nonexistent,93.994,-36.4,4.857,5191.0,0


#### job OHE,loan OHE, marital OHE,month OHE, poutcome OHE


contact,previous,housing,loan

In [20]:
encoder3 = TargetEncoder(cols=['job'])
df = encoder3.fit_transform(df,df['y'])

#encoder4 = BinaryEncoder(cols=['loan'])
#df = encoder4.fit_transform(df)

'''
encoder5 = TargetEncoder(cols=['marital'])
df = encoder5.fit_transform(df,df['y'])
'''
encoder6 = TargetEncoder(cols=['month'])
df = encoder6.fit_transform(df,df['y'])

encoder7 = TargetEncoder(cols=['poutcome'])
df = encoder7.fit_transform(df,df['y'])

df.head()

,age,job,education,month,duration,campaign,poutcome,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,0.097537,0.102477,0.064231,4.0,1,0.087582,93.994,-36.4,4.857,5191.0,0
1,57,0.079706,0.108162,0.064231,2.0,1,0.087582,93.994,-36.4,4.857,5191.0,0
2,37,0.079706,0.108162,0.064231,4.0,1,0.087582,93.994,-36.4,4.857,5191.0,0
3,40,0.128756,0.082047,0.064231,3.0,1,0.087582,93.994,-36.4,4.857,5191.0,0
4,56,0.079706,0.108162,0.064231,5.0,1,0.087582,93.994,-36.4,4.857,5191.0,0


In [21]:
df.describe()

,age,job,education,month,duration,campaign,poutcome,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
count,39183.000000,39183.000000,39183.000000,39183.000000,39183.000000,39183.000000,39183.000000,39183.000000,39183.000000,39183.000000,39183.000000,39183.00000
mean,39.858638,0.110890,0.110862,0.110890,4.252431,2.539826,0.110890,93.570899,-40.537358,3.621468,5167.326292,0.11089
std,10.291618,0.045345,0.020573,0.086550,4.100724,2.600242,0.099651,0.577196,4.624393,1.731118,71.803554,0.31400
min,17.000000,0.069225,0.078115,0.064231,0.000000,1.000000,0.087582,92.201000,-50.800000,0.634000,4963.600000,0.00000
25%,32.000000,0.079706,0.102477,0.064231,2.000000,1.000000,0.087582,93.075000,-42.700000,1.344000,5099.100000,0.00000
50%,38.000000,0.107959,0.108162,0.089764,3.000000,2.000000,0.087582,93.444000,-41.800000,4.857000,5191.000000,0.00000
75%,47.000000,0.128756,0.136292,0.103512,5.000000,3.000000,0.087582,93.994000,-36.400000,4.961000,5228.100000,0.00000
max,98.000000,0.304102,0.161008,0.504912,35.000000,30.000000,0.648352,94.767000,-26.900000,5.045000,5228.100000,1.00000


#### train test SPLIT

In [22]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import ADASYN,SMOTE   

X = df.drop('y', axis=1)  
y = df['y']   

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y            
)


'''
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
'''

'''
from sklearn.preprocessing import MinMaxScaler
minmax = MinMaxScaler()
X_train = minmax.fit_transform(X_train)
X_test = minmax.transform(X_test)
'''

ismote = SMOTE(sampling_strategy = "auto", 
                k_neighbors = 5)   

X_train_res, y_train_res  = ismote.fit_resample(X_train, y_train)

from collections import Counter

print(Counter(y_train_res))


Counter({0: 24387, 1: 24387})


### contact,previous,housing,loan

In [23]:
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import nevergrad as ng

def evaluate_nb(log10_var_smoothing):
    """
    Evaluates GaussianNB via cross-validation.
    Nevergrad minimizes this function => return negative mean accuracy.
    We optimize log10(var_smoothing) to search on a log scale.
    """
    var_smoothing = 10.0 ** log10_var_smoothing
    clf = GaussianNB(var_smoothing=var_smoothing)
    scores = cross_val_score(clf, X_train, y_train, cv=5, scoring='accuracy')
    return -np.mean(scores)

param_space = ng.p.Instrumentation(
    log10_var_smoothing=ng.p.Scalar(lower=-12, upper=-6)
)

optimizer = ng.optimizers.OnePlusOne(parametrization=param_space, budget=50)
best_suggestion = optimizer.minimize(lambda *a, **k: evaluate_nb(*a, **k))
best_log10_vs = float(best_suggestion.kwargs['log10_var_smoothing'])
best_var_smoothing = 10.0 ** best_log10_vs

print("\n--- Optimization Results (Naive Bayes) ---")
print({"var_smoothing": best_var_smoothing})

nb_best = GaussianNB(var_smoothing=best_var_smoothing)
nb_best.fit(X_train, y_train)

y_pred_nb = nb_best.predict(X_test)

print("\n--- Final Model Performance on Test Set (Naive Bayes) ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_nb):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))



--- Optimization Results (Naive Bayes) ---
{'var_smoothing': 4.778554370063398e-07}

--- Final Model Performance on Test Set (Naive Bayes) ---
Accuracy: 0.8873

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.93      0.94     10451
           1       0.49      0.54      0.52      1304

    accuracy                           0.89     11755
   macro avg       0.72      0.74      0.73     11755
weighted avg       0.89      0.89      0.89     11755


Confusion Matrix:
[[9721  730]
 [ 595  709]]


#### train-test

In [24]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# --- 5. Train Final Model (Naive Bayes) ---
nb_best = GaussianNB(var_smoothing=best_var_smoothing)
nb_best.fit(X_train, y_train)

# --- 6. Evaluate on Test Set ---
y_pred = nb_best.predict(X_test)  

print("\n--- Final Model Performance on Test Set (Naive Bayes) ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


# import pandas as pd 

# feature_importances = pd.Series(clf.feature_importances_, index=X.columns)
# feature_importances = feature_importances.sort_values(ascending=False)

# print("\nFeature Importances:")
# print(feature_importances)



--- Final Model Performance on Test Set (Naive Bayes) ---
Accuracy: 0.8873

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.93      0.94     10451
           1       0.49      0.54      0.52      1304

    accuracy                           0.89     11755
   macro avg       0.72      0.74      0.73     11755
weighted avg       0.89      0.89      0.89     11755


Confusion Matrix:
[[9721  730]
 [ 595  709]]


In [25]:
import joblib

# --- TAHAP PENYIMPANAN (EXPORT) ---
# Pastikan variabel 'nb_best', 'ohe', 'encoder3', dll sudah dijalankan di cell atasnya

artifacts = {
    'model': nb_best,           # Model Naive Bayes terbaik Anda
    'encoders': {
        'education': ohe,       # Encoder untuk kolom Education
        'job': encoder3,        # Encoder untuk kolom Job
        'month': encoder6,      # Encoder untuk kolom Month
        'poutcome': encoder7    # Encoder untuk kolom Poutcome
    }
}

# Simpan ke file .pkl
model_filename = 'model_lead_scoring.pkl'
joblib.dump(artifacts, model_filename)

print(f"SUKSES! Model disimpan sebagai: {model_filename}")

SUKSES! Model disimpan sebagai: model_lead_scoring.pkl
